In [ ]:
!pip install duckdb --upgrade
!pip install deltalake

In [2]:
from   datetime          import datetime
from   deltalake.writer  import write_deltalake
from   shutil            import unpack_archive
from   urllib.request    import urlopen
from   psutil            import *
import duckdb
import time
import glob
import os
import re
import requests
import multiprocessing
import json

In [3]:
total_files   = 2
Source        = "/content/Files/zip/"
Destination   = "/content/Files/csv/"
table_path    = '/content/dbo/dd'

In [4]:
con = duckdb.connect()
con.sql(" force install delta from core; update extensions")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌────────────────┬────────────┬─────────────────────┬──────────────────┬─────────────────┐
│ extension_name │ repository │    update_result    │ previous_version │ current_version │
│    varchar     │  varchar   │       varchar       │     varchar      │     varchar     │
├────────────────┼────────────┼─────────────────────┼──────────────────┼─────────────────┤
│ delta          │ core       │ NO_UPDATE_AVAILABLE │ 5fb3d7c          │ 5fb3d7c         │
└────────────────┴────────────┴─────────────────────┴──────────────────┴─────────────────┘

In [5]:
def get_all_zip_files_api(owner, repo, path):
    """Use GitHub API to recursively find all zip files"""
    all_zips = []
    api_url = f'https://api.github.com/repos/{owner}/{repo}/contents/{path}'

    try:
        response = requests.get(api_url)
        response.raise_for_status()
        items = response.json()

        for item in items:
            if item['type'] == 'dir':
                # Recursively get files from subdirectories
                subdir_zips = get_all_zip_files_api(owner, repo, item['path'])
                all_zips.extend(subdir_zips)
            elif item['type'] == 'file' and item['name'].endswith('.zip'):
                # Add the download URL for this zip file
                all_zips.append({
                    'name': item['name'],
                    'path': item['path'],
                    'download_url': item['download_url']
                })
    except Exception as e:
        print(f"Error accessing {api_url}: {e}")

    return all_zips

def download(Path, total_files):
    if not os.path.exists(Path):
        os.makedirs(Path, exist_ok=True)

    # Get all zip files using GitHub API
    print("Fetching file list from GitHub...")

    current = [os.path.basename(x) for x in glob.glob(os.path.join(Path, '*.zip'))]

    # Check if folder already has the requested number of files
    if len(current) >= total_files:
        print(f"Folder already has {len(current)} file(s). No download needed.")
        return "done"

    filelist = get_all_zip_files_api('djouallah', 'fabric_demo', 'data/archive')
    filelist = sorted(filelist, key=lambda x: x['name'], reverse=True)

    # Calculate how many more files are needed
    files_needed = total_files - len(current)
    files_to_upload = [f for f in filelist if f['name'] not in current][:files_needed]

    print(f"{len(files_to_upload)} New File(s) to Download")

    if len(files_to_upload) > 0:
        for file_info in files_to_upload:
            filename = file_info['name']
            download_url = file_info['download_url']
            print(f"Downloading {filename}...")

            try:
                with requests.get(download_url, stream=True) as resp:
                    resp.raise_for_status()
                    with open(os.path.join(Path, filename), "wb") as f:
                        for chunk in resp.iter_content(chunk_size=8192):
                            f.write(chunk)
                print(f"  ✓ {filename} downloaded successfully")
            except Exception as e:
                print(f"  ✗ Failed to download {filename}: {e}")

    return "done"

In [6]:
def uncompress(x):
        unpack_archive(str(Source+x), str(Destination), 'zip')

def unzip(Source, Destination, Nbr_Files_to_Download):
    if not os.path.exists(Destination):
      os.makedirs(Destination, exist_ok=True)
    filelist=[os.path.basename(x) for x in glob.glob(Source+'*.zip')]
    ### check the unzipped files already
    current = [os.path.basename(x) for x in glob.glob(Destination+'*.CSV')]

    # Skip if we already have enough files
    if len(current) >= Nbr_Files_to_Download:
      print(f'Already have {len(current)} files, skipping extraction')
      return "nothing to see here"

    current = [w.replace('.CSV','.zip') for w in current]
    #unzip only the delta
    remaining = Nbr_Files_to_Download - len(current)
    files_to_upload = list(set(filelist) - set(current))[:remaining]
    files_to_upload = list(dict.fromkeys(files_to_upload))
    print(str(len(files_to_upload)) + ' New File uncompressed')
    if len(files_to_upload) != 0 :
      with multiprocessing.Pool() as pool:
       for _ in pool.imap_unordered(uncompress, files_to_upload, chunksize=1):
         pass
      return "done"
    else:
     return "nothing to see here"

In [ ]:
download(Source, total_files)
unzip(Source, Destination, total_files)

Fetching file list from GitHub...
2 New File(s) to Download
  ✓ PUBLIC_DAILY_202603240000_20260325040503.zip downloaded successfully
  ✓ PUBLIC_DAILY_202603230000_20260324040503.zip downloaded successfully
2 New File uncompressed


In [ ]:
con.sql(F"""
    SET preserve_insertion_order = false;
    create or replace view source as
    WITH raw AS (
        SELECT * FROM read_csv('{Destination}PUBLIC_DAILY*.CSV',
            skip=1,
            header=0,
            all_varchar=1,
            columns={{
                'I': 'VARCHAR','UNIT': 'VARCHAR','XX': 'VARCHAR','VERSION': 'VARCHAR','SETTLEMENTDATE': 'VARCHAR','RUNNO': 'VARCHAR',
                'DUID': 'VARCHAR','INTERVENTION': 'VARCHAR','DISPATCHMODE': 'VARCHAR','AGCSTATUS': 'VARCHAR','INITIALMW': 'VARCHAR',
                'TOTALCLEARED': 'VARCHAR','RAMPDOWNRATE': 'VARCHAR','RAMPUPRATE': 'VARCHAR','LOWER5MIN': 'VARCHAR',
                'LOWER60SEC': 'VARCHAR','LOWER6SEC': 'VARCHAR','RAISE5MIN': 'VARCHAR','RAISE60SEC': 'VARCHAR',
                'RAISE6SEC': 'VARCHAR','MARGINAL5MINVALUE': 'VARCHAR','MARGINAL60SECVALUE': 'VARCHAR',
                'MARGINAL6SECVALUE': 'VARCHAR','MARGINALVALUE': 'VARCHAR','VIOLATION5MINDEGREE': 'VARCHAR',
                'VIOLATION60SECDEGREE': 'VARCHAR','VIOLATION6SECDEGREE': 'VARCHAR','VIOLATIONDEGREE': 'VARCHAR',
                'LOWERREG': 'VARCHAR','RAISEREG': 'VARCHAR','AVAILABILITY': 'VARCHAR','RAISE6SECFLAGS': 'VARCHAR',
                'RAISE60SECFLAGS': 'VARCHAR','RAISE5MINFLAGS': 'VARCHAR','RAISEREGFLAGS': 'VARCHAR',
                'LOWER6SECFLAGS': 'VARCHAR','LOWER60SECFLAGS': 'VARCHAR','LOWER5MINFLAGS': 'VARCHAR',
                'LOWERREGFLAGS': 'VARCHAR','RAISEREGAVAILABILITY': 'VARCHAR','RAISEREGENABLEMENTMAX': 'VARCHAR',
                'RAISEREGENABLEMENTMIN': 'VARCHAR','LOWERREGAVAILABILITY': 'VARCHAR','LOWERREGENABLEMENTMAX': 'VARCHAR',
                'LOWERREGENABLEMENTMIN': 'VARCHAR','RAISE6SECACTUALAVAILABILITY': 'VARCHAR',
                'RAISE60SECACTUALAVAILABILITY': 'VARCHAR','RAISE5MINACTUALAVAILABILITY': 'VARCHAR',
                'RAISEREGACTUALAVAILABILITY': 'VARCHAR','LOWER6SECACTUALAVAILABILITY': 'VARCHAR',
                'LOWER60SECACTUALAVAILABILITY': 'VARCHAR','LOWER5MINACTUALAVAILABILITY': 'VARCHAR','LOWERREGACTUALAVAILABILITY': 'VARCHAR'
            }},
            filename=1,
            null_padding=true,
            ignore_errors=1,
            auto_detect=false)
			WHERE I='D' AND UNIT='DUNIT' AND VERSION='3'
    )
    SELECT
        UNIT,
        DUID,
        COLUMNS(*EXCLUDE(DUID, UNIT, SETTLEMENTDATE,filename,I,XX))::DOUBLE,
		filename as file,
		SETTLEMENTDATE::TIMESTAMPTZ AS SETTLEMENTDATE,
        SETTLEMENTDATE::DATE AS date,
        YEAR(SETTLEMENTDATE::TIMESTAMP) AS year
    FROM raw
""")

In [ ]:
con.sql(" describe source ").show(max_rows=100)

┌──────────────────────────────┬──────────────────────────┬─────────┬─────────┬─────────┬─────────┐
│         column_name          │       column_type        │  null   │   key   │ default │  extra  │
│           varchar            │         varchar          │ varchar │ varchar │ varchar │ varchar │
├──────────────────────────────┼──────────────────────────┼─────────┼─────────┼─────────┼─────────┤
│ UNIT                         │ VARCHAR                  │ YES     │ NULL    │ NULL    │ NULL    │
│ DUID                         │ VARCHAR                  │ YES     │ NULL    │ NULL    │ NULL    │
│ VERSION                      │ DOUBLE                   │ YES     │ NULL    │ NULL    │ NULL    │
│ RUNNO                        │ DOUBLE                   │ YES     │ NULL    │ NULL    │ NULL    │
│ INTERVENTION                 │ DOUBLE                   │ YES     │ NULL    │ NULL    │ NULL    │
│ DISPATCHMODE                 │ DOUBLE                   │ YES     │ NULL    │ NULL    │ NULL    │


In [ ]:
write_deltalake(f"{table_path}",con.sql(" select * from source limit 0 ").arrow(),mode="ignore")

In [ ]:
con.sql(f" describe from delta_scan('{table_path}')").show(max_rows=100)

┌──────────────────────────────┬──────────────────────────┬─────────┬─────────┬─────────┬─────────┐
│         column_name          │       column_type        │  null   │   key   │ default │  extra  │
│           varchar            │         varchar          │ varchar │ varchar │ varchar │ varchar │
├──────────────────────────────┼──────────────────────────┼─────────┼─────────┼─────────┼─────────┤
│ UNIT                         │ VARCHAR                  │ YES     │ NULL    │ NULL    │ NULL    │
│ DUID                         │ VARCHAR                  │ YES     │ NULL    │ NULL    │ NULL    │
│ VERSION                      │ DOUBLE                   │ YES     │ NULL    │ NULL    │ NULL    │
│ RUNNO                        │ DOUBLE                   │ YES     │ NULL    │ NULL    │ NULL    │
│ INTERVENTION                 │ DOUBLE                   │ YES     │ NULL    │ NULL    │ NULL    │
│ DISPATCHMODE                 │ DOUBLE                   │ YES     │ NULL    │ NULL    │ NULL    │


In [ ]:
con.sql(F"""
    copy source to xx.parquet
""")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

In [ ]:
con.sql(F"""
    describe from xx.parquet
""").show(max_rows=100)

┌──────────────────────────────┬──────────────────────────┬─────────┬─────────┬─────────┬─────────┐
│         column_name          │       column_type        │  null   │   key   │ default │  extra  │
│           varchar            │         varchar          │ varchar │ varchar │ varchar │ varchar │
├──────────────────────────────┼──────────────────────────┼─────────┼─────────┼─────────┼─────────┤
│ UNIT                         │ VARCHAR                  │ YES     │ NULL    │ NULL    │ NULL    │
│ DUID                         │ VARCHAR                  │ YES     │ NULL    │ NULL    │ NULL    │
│ VERSION                      │ DOUBLE                   │ YES     │ NULL    │ NULL    │ NULL    │
│ RUNNO                        │ DOUBLE                   │ YES     │ NULL    │ NULL    │ NULL    │
│ INTERVENTION                 │ DOUBLE                   │ YES     │ NULL    │ NULL    │ NULL    │
│ DISPATCHMODE                 │ DOUBLE                   │ YES     │ NULL    │ NULL    │ NULL    │


In [ ]:

con.sql(F"""
    attach or replace '{table_path}' as destination (type delta,read_only false ) ;
    INSERT INTO destination by name SELECT * FROM source
""")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

TransactionException: TransactionContext Error: Failed to commit: Failed to cast value: Could not convert string '' to DOUBLE